In [1]:
import sys
import os
from pathlib import Path
import pandas as pd
import requests
import io
from dotenv import load_dotenv
import openpyxl
from urllib.parse import quote

load_dotenv()
DIR = Path().resolve().parent
if str(DIR) not in sys.path:
    sys.path.append(str(DIR))

from src.performance_dashboard.services.sharepoint_client import get_folder_items, get_file_content, get_sharepoint_file
from src.performance_dashboard.utils import get_secret

In [2]:
path1 = "opdrachtgever_1.parquet"
path2 = "peer_1.parquet"
sub_folder = get_secret("TRANSFORMED_RESPONSES_FOLDER")

hr_folder = get_secret("HR_CYCLE_FOLDER")

    # Pad opbouwen
file_path = f"{hr_folder}/{sub_folder}/{path2}" if sub_folder else f"{hr_folder}/{path2}"
encoded_path = quote(file_path, safe='/')

In [3]:
items, headers, site_id = get_folder_items()

✅ Connected with site: Veneficus Data Safe
✅ Found Site ID: veneficus.sharepoint.com,e7754da8-bfb5-427b-bae9-2daa9d0eacbc,3b7a57d3-1d9a-4263-9719-aa56cea0efb7

📁 Content of map 'HR Cycle':
   📁 Naam: TypeformData
      URL: https://veneficus.sharepoint.com/sites/VeneficusDataSafe/Gedeelde%20documenten/HR%20Cycle/TypeformData
   📄 Naam: Testbestan.xlsx
      URL: https://veneficus.sharepoint.com/sites/VeneficusDataSafe/_layouts/15/Doc.aspx?sourcedoc=%7B49BD27D0-9AAF-42B9-A292-5F4D741D1DFD%7D&file=Testbestan.xlsx&action=default&mobileredirect=true
   📄 Naam: Werknemers_gegevens - Test.xlsx
      URL: https://veneficus.sharepoint.com/sites/VeneficusDataSafe/_layouts/15/Doc.aspx?sourcedoc=%7BD01E5630-2F37-4476-AA36-5BFECEB80C33%7D&file=Werknemers_gegevens%20-%20Test.xlsx&action=default&mobileredirect=true
   📄 Naam: Werknemers_gegevens.xlsx
      URL: https://veneficus.sharepoint.com/sites/VeneficusDataSafe/_layouts/15/Doc.aspx?sourcedoc=%7B144D491D-5701-47A0-81E2-E75F9C99EF74%7D&file=Werk

In [4]:
download_url = f"https://graph.microsoft.com/v1.0/sites/{site_id}/drive/root:/{encoded_path}:/content" 

print(f"DEBUG: Full URL: {download_url}")
response = requests.get(download_url, headers=headers)
file_bytes = response.content

DEBUG: Full URL: https://graph.microsoft.com/v1.0/sites/veneficus.sharepoint.com,e7754da8-bfb5-427b-bae9-2daa9d0eacbc,3b7a57d3-1d9a-4263-9719-aa56cea0efb7/drive/root:/HR%20Cycle/TypeformData/transformed_responses/peer_1.parquet:/content


In [5]:
file_bytes = get_file_content(headers, site_id, encoded_path)

DEBUG: Full URL: https://graph.microsoft.com/v1.0/sites/veneficus.sharepoint.com,e7754da8-bfb5-427b-bae9-2daa9d0eacbc,3b7a57d3-1d9a-4263-9719-aa56cea0efb7/drive/root:/HR%20Cycle/TypeformData/transformed_responses/peer_1.parquet:/content


In [6]:
if file_bytes:
        try: # Check extensie voor de juiste pandas reader
            if path1.endswith('.parquet'):
                print(pd.read_parquet(io.BytesIO(file_bytes)))
            else:
                print("not a parquet")
        except Exception as e:
            print(f"❌ Error when reading DataFrame {path1}: {e}")

    form_id form_type                    response_token          submitted_at  \
0  dSDKPEzu      quiz  hr0kyf8m6f2htadzz1hr0kyf8mc0awh0  2026-01-20T09:04:20Z   
1  dSDKPEzu      quiz  hr0kyf8m6f2htadzz1hr0kyf8mc0awh0  2026-01-20T09:04:20Z   
2  dSDKPEzu      quiz  hr0kyf8m6f2htadzz1hr0kyf8mc0awh0  2026-01-20T09:04:20Z   
3  dSDKPEzu      quiz  hr0kyf8m6f2htadzz1hr0kyf8mc0awh0  2026-01-20T09:04:20Z   
4  dSDKPEzu      quiz  hr0kyf8m6f2htadzz1hr0kyf8mc0awh0  2026-01-20T09:04:20Z   
5  dSDKPEzu      quiz  hr0kyf8m6f2htadzz1hr0kyf8mc0awh0  2026-01-20T09:04:20Z   
6  dSDKPEzu      quiz  hr0kyf8m6f2htadzz1hr0kyf8mc0awh0  2026-01-20T09:04:20Z   
7  dSDKPEzu      quiz  hr0kyf8m6f2htadzz1hr0kyf8mc0awh0  2026-01-20T09:04:20Z   

   employee_name  respondent_name   question_id    question_type  \
0  employee_name  respondent_name  zEFBRre0JvuV  multiple_choice   
1  employee_name  respondent_name  xrpnryUhvlwC  multiple_choice   
2  employee_name  respondent_name  tNNOU1qabV2K  multiple_choice  